# Reproduction pipeline

Drag `tennis_pipeline.tar.gz` into the Files pane on the left, wait for the upload to finish, then run the cells in order. About two hours for everything; see the notes on each section for what can be skipped.

The distribution ships no precomputed results, so every number you see is one you produced. Eras are defined once in `tennisdom/config.py` as four blocks of twelve years, and every script reads them from there.

`02` must run before `03`: it writes `out/02_hyper.json`, which `config.py` then reads in preference to its defaults. The first cell prints which values are in use.

In [ ]:
!tar xzf /content/tennis_pipeline.tar.gz -C /content
%cd /content/pipeline/scripts
!python -c "import sys; sys.path.insert(0,'..'); from tennisdom import config as C; print('eras:', C.ERAS); print(C.describe())"

### Fit and estimands, about 18 minutes

In [ ]:
!python 03_posterior.py
!python 04_estimands.py

### Everything that depends on the eras

In [ ]:
!python 08_sensitivity.py
!python 09_trajectories.py

### Surfaces

Independent of the era division, so skip this if you still have `out/05_surfaces.pkl` from a previous run.

In [ ]:
!python 05_surfaces.py

### Check, tables, era comparison, rolling windows

### The two numerical studies

Neither depends on the era division, so skip both if you still have `out/sim3.csv` and the Appendix B numbers from a previous run. `06` produces Tables 1 and 2 and takes about 25 minutes; `07` produces Table 3 and takes about 20.

In [ ]:
!python 06_simulation.py

In [ ]:
!python 07_validation.py

In [ ]:
!python 13_check.py
!python 14_tables.py
!python 16_eras.py
!python 15_rolling.py

### Best decade against best other decade

Finds the ten-year window with the most concurrent dominance, then the best window that does not overlap it, and compares them on every summary together with who held the top three in each. Needs `09_trajectories.py` to have run, for the occupancy columns.

In [ ]:
!python 17_bestwindows.py

In [ ]:
import glob
for f in sorted(glob.glob('../out/tables/*.tex')):
    print('='*70); print(f.split('/')[-1]); print('='*70); print(open(f).read())

### Figures, then download

In [ ]:
!python 10_figures_main.py
!python 11_figure_persistence.py
!python 12_figure_trajectories.py
!ls ../out/*.pdf

In [ ]:
import shutil, glob, os
os.makedirs('/content/res/tables', exist_ok=True)
for f in glob.glob('../out/*.pdf')+glob.glob('../out/*.csv')+glob.glob('../out/*.json'):
    shutil.copy2(f,'/content/res')
for f in glob.glob('../out/tables/*.tex'): shutil.copy2(f,'/content/res/tables')
shutil.make_archive('/content/era_rerun','zip','/content/res')
from google.colab import files; files.download('/content/era_rerun.zip')

---

### Optional: refit the hyperparameters

`config.py` already holds the values `02` selects, so this is a verification rather than a requirement, and it takes about 35 minutes. Running it writes `out/02_hyper.json`, which `config.py` then reads in preference to its defaults; that also removes the small rounding difference between $\tau=0.0473$ and the fitted $0.047290$. If you run it, rerun `03` and everything after it.

In [ ]:
!python 01_data.py
!python 02_hyperparameters.py